## Zagon računalnika

Ob prvem zagonu se na procesorju PŠ (pc) nastavi na:
- Intel x86: 0xFFFFFFF0
- PowerPC: 0xFFF00100 ali 0x00000100
- ARM: 0x00000000 ali 0xFFFF0000

## BIOS
- koda, ki se začne izvajati ob zagonu
- gonilnik za V/I enote
**Koda se z ginilnikom V/I poveže na disk, ... in pridobi PROGRAM: operaciski sistem**
- *master boot record, MBR* **Ni nujno, da je že prvi program OS, lahko je šela ta nalagalnik**

Delimo jih na:  
- Klasični **BIOS** (Starejši)
    - 16-bitni način delovanja
    - omejitve pri particijah (MBR)
    - brez grafičnega vmesnika
    - osnovno nastavljanje strojne opreme
- **UEFI** (najbolj priljubljendanes)
    - podpira velike diske (GPT)
    - grafični vmesnik
    - hitrejši zagon
    - Secure Boot
    - modularen in razširljiv firmware
- **Vendor-specific** BIOS/UEFI implementacije
    - Proizvajalci imajo svoje lastne BIOS/UEFI, ki se razlikujejo po funkcijah in grafiki, a vsi temeljijo na BIOS ali UEFI standardu.
    - [Najbloj_pogosti](./Ostalo_Ideje.ipynb)

### Secure BOOT
- Secure Boot ščiti zagon sistema tako, da dovoli nalaganje samo kriptografsko podpisane in zaupanja vredne zagonske programske opreme (bootloader, OS kernel, EFI aplikacije).
- **TFTP in UDB nimata zveze s Secure Boot.**

## Bootloader
Program, ki se naloži na računalnik pri zagonu in nam omogoča nalaganje OS.
- GRUB, LILO, U-Boot, Windows Boot Maneger,...
**Lahko ga prenese preko diska ali omrežja (strežnika)**

- **Nalaganje preko omrežja:**
- ne potrebujemo diska na računalniku
- *OS preprosto zamenjamo za vse rač. saj ga žamenjamo samo na stežniku*
- počasneje, bolj ranljivo (TFTP)
    1. znati poiskati strežnik, s katerega bo naložil OS
    2. znati se nastaviti, kot bo svetoval/zahteval strežnik
    3. prenesti OS k sebi
    4. namestiti OS in ga zagnati
*1. 2. -> bootp*
*3. -> tftp*

## BOOTP
[Izgled_paketa1](./Slike/Boofp_packet.png)  
[Izgled_paketa2](./Slike/Boofp_packet1.png)
- Na prenosni plasti uporablja nepovezavni načun - UDP protokol
- Lahko ja hkrati več strežnikov in več odjemalcev
1. Odjemalec pošlje Brotcast (ne pozna IP strežnika) (podatkovna plast: Lokalno omrežje)
2. Strežnik mu dodeli IP in sporoči mu, kje se nahaja odjemalčev OS (ni nujno lokalno)

### Oprema:
Starejša FreeBSD: (ročna konfiguracija v [/etc/bootptab](./Ukazi-Faili.ipynb))
- bootpd
- bootpgw


## TFTP
[Izgled_paketa](./Slike/TFTP_packet.png)
Datoteke, ki jih streže so v [/tftpboot](./Ukazi-Faili.ipynb)
Telo poenostavljena verzija FTP protokola (moznost prenosa podatkov)
- Največji paket do 1TB
**Uporabljen protokol UDP** (preptostost, majhna glava(Manjši paketi), TCP ne podpira broadcastov(preden imamo IP))
- Lažja je tudi implementacija v mrežni kartici

### Delovanje:
1. odjemalec pošlje zahtevo po branju
2. strežnik odgovori z DATA paketom (določi vrata)
3. za vsak paket odjemalec odgovori (ACK paket) (strežnik pošilja dokler ne prejme odgovora)
4. **Zadnji paket je manjši od ostalih** (tako vemo, da je zadnji)


## DNS
!!Dodatni podatki: IANA !!
**Uporablja UDP protokol** (UDP: v glavi označimo 17)
*ESP-Encapsulating Security Payload* (v glavi označimo 50)
- oznake v glavi dobimo [/etc/protocols](./Ukazi-Faili.ipynb)

DHCP: IP, prevzeti prehod, DNS strežnik(mora poznati root DNS strežnik, če slučajno ne pozna naslova)
- domen name vrata :53 (datoteka [/etc/services](./Ukazi-Faili.ipynb))
- IP root strežnikov [/etc/namedb/named.root](./Ukazi-Faili.ipynb)

## Priklop na omrežje (DHCP*)
- Stalna IP številka (samo na stacionarnih računalnikih)
- Ponudniki želijo poslužiti več strank, kot imajo IP naslovov
- **Bootp protokol v prvem koraku odjemalcu pošlje tudi podatke za nastavitev IP naslova** (, da mu podatke za DHCP)
+ Pridobiti moramo:
    - IP naslov
    - Naslov prehoda
    - Naslov DNS strežnika
    - Naslov vmesnega strežnika (proxy) (v primeru, da je strežnik z OS zunaj naše lokalne mreže)
    - **Vend** (polje ostalo v paketu):
        - Prva vrednost je magic cookie (4 biti: 99.130.83.99) (pove nam, da je za njim nahajajo BOOTp options → kasneje DHCP options)
        - Dve vrsti polj: (po dolžini) (značke 128-254: lokalne razširitve)
            - stalna: *zalog 1: značka [podatki]* npr: (značka: 1, podatki: 4 zlogi): 1.255.255.255.0
            - spremenljiva: *zlog 1: značka, zlog 2: dolžina podatkov, ostali zlogi: podatki* 
                npr. (značka: 3, podatki: N/4 naslovov): 3.4.1.2.3.4

    